# Global 2nd-Tier Metals Transparency & Cost-at-Risk Engine

**Portfolio piece for: an electrical-equipment manufacturer (circuit breakers, contactors, switchgear)**
**Role: Global 2nd-Tier Manager (f/m/d) – Data & AI Integration**

---

### The thesis

A dashboard tells you *what you spent*. This engine tells you **what you should have spent, who you are really exposed to, and what it will cost you next quarter.**

Three claims, each proven with a number later in this notebook:

1. **Transparency** – a Tier-2 network that looks diversified is actually concentrated, and only entity resolution can see it.
2. **Cost** – a supplier's "metal cost increase" can be tested against BOM × index × FX arithmetic, turning a price letter into a negotiation position.
3. **Risk** – contract clause design, geopolitics and supplier cyber maturity can be priced in euros, not adjectives.

### How this notebook is organised

| Part | Content |
|---|---|
| **Part 0** | Setup and data inventory |
| **Part 1** | *Phase 1 rebuilt step by step* – how the synthetic dataset was designed and why |
| **Part 2** | The commercial analytics: scenarios S5, S3, S1, S7 quantified in euros |
| **Part 3** | *Phase 2 rebuilt step by step* – the entity resolution engine, from normalisation to evaluation |
| **Part 4** | Scenario S2 – the hidden single source, plus S4 and S6 |
| **Part 5** | The executive summary, and what Phases 3–5 add |

## What has been built so far — the full story from scratch

Before any code: here is the honest chronology of the project, so you can follow *why* each artefact exists.

### Step 1 — Read the job advert as a requirements document

Seven responsibilities were listed in the advert. Each one was turned into a *scenario that must be discoverable in data*, rather than a bullet point on a slide:

| Job advert responsibility | Scenario engineered into the data |
|---|---|
| "structure, analyse and maintain global 2nd-tier supplier data" | **S6** – 28% of Tier-1 metal spend has no declared Tier-2 at all |
| "ensure transparency within global 2nd-tier networks" | **S2** – one Chinese smelter hides behind 3 spellings and 6 Tier-1s |
| "bring cost and price transparency… analysis of cost drivers, market indices, price trends" | **S3** – an 11.5% surcharge claim that BOM × index maths cannot justify |
| "sourcing strategies… contract management… total cost optimization" | **S1** – fixed-price copper contracts expiring after a 38% price ramp<br>**S7** – caps and lags that block favourable index moves |
| "material availability… aluminium, copper and steel… resilience" | **12 disruption events** that actually lengthen purchase-order lead times |
| "cybersecurity… suppliers comply to security standards" | **S4** – the two largest Tier-2s have the weakest cyber posture |
| "market, contractual and geopolitical challenges" | **S5** – USD strength amplifies a copper rally for a EUR-reporting buyer |

### Step 2 — Design a relational schema, not a flat file

15 tables in a star schema: 8 dimensions, 6 facts, 1 ground-truth answer key. Every fact resolves to a dimension; no orphan keys. The grain of the largest fact table is one purchase-order line.

Deliberate design choices worth defending in an interview:

- **`bom_metal_content.cost_share`** is the hinge of the entire model. It converts "we bought a circuit breaker for €47" into "€12.80 of that was copper". Without it, no should-cost model is possible.
- **`dim_contract`** encodes real clause mechanics — `passthrough_pct`, `reset_frequency`, `reset_lag_months`, `cap_pct`, `floor_pct` — so prices are *derived*, not random. This is what lets us later prove that a clause leaked value.
- **`dim_supplier_tier2_raw.duns_number` is deliberately empty.** No hard key exists, which forces genuine name-based matching. That mirrors reality: Tier-1s send you spreadsheets, not D&B identifiers.
- **Fiscal calendar** (October–September, common in German industrial companies) is built into `dim_date`.

### Step 3 — Generate the data with causality, not noise

`scripts/generate_mock_data.py` — one seeded RNG, fully reproducible. Prices are not drawn from a distribution; they are **computed**:

```
unit_price = base_price
             × (1 + clip(passthrough_pct × cost_share × index_delta_at_lagged_reset,
                          floor_pct, cap_pct))
             × non_metal_inflation
             × small_noise
```

Market indices combine **scripted regimes** (a copper ramp, an aluminium energy crisis, a steel decline, a nickel export-ban spike) with geometric Brownian noise, so the series look real *and* tell a story. Disruption events feed back into `lead_time_actual_days`. FX is engineered so the USD rally overlaps the copper rally.

### Step 4 — Make the generator self-auditing

The script ends with ~25 assertions: referential integrity, domain sanity (no negative prices, unique keys), and **scenario assertions** — the S3 negotiation gap must exceed 5 percentage points, S6 coverage must land between 62% and 82%, the EUR copper move must exceed the USD move. If a future change breaks the story, the script exits non-zero.

### Step 5 — Build the entity resolution engine

`scripts/resolve_tier2_entities.py` collapses 48 dirty supplier records into 20 golden entities, using a two-pass design (deterministic normalised key, then a probabilistic ensemble), with blocking for scalability and a human-review queue for borderline pairs. It scores itself against the withheld ground truth: **precision 100%, recall 100%, 20/20 clusters exactly reconstructed.**

Every string metric — Levenshtein, Jaro-Winkler, token-set ratio, Dice — is implemented from scratch, so there is no library to hide behind and the logic runs in any Spark or Fabric environment without extra packages.

**This notebook now re-walks both phases, executing the key logic live.**

---
# Part 0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# locate the project root regardless of where the kernel starts
PROJECT = Path.cwd()
while not (PROJECT / "data" / "raw").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
SCRIPTS = PROJECT / "scripts"
sys.path.insert(0, str(SCRIPTS))


def load(name, folder=None):
    folder = RAW if folder is None else folder
    return pd.read_csv(folder / f"{name}.csv", encoding="utf-8-sig")


print("project root :", PROJECT)
print("raw tables   :", len(list(RAW.glob("*.csv"))))
print("processed    :", len(list(PROCESSED.glob("*.csv"))))

---
# Part 1 — Phase 1: the synthetic dataset

## 1.1 The table inventory

Fifteen tables. Note the grain of each one: this is the first thing an interviewer will check, because a wrong grain silently double-counts spend.

In [ ]:
GRAIN = {
    "dim_date":                 "1 row per calendar day",
    "dim_metal":                "1 row per traded metal",
    "fact_market_index":        "metal x month",
    "fact_fx_rate":             "currency x month",
    "dim_part":                 "1 row per purchased part",
    "bom_metal_content":        "part x metal   <- the hinge of the model",
    "dim_supplier_tier1":       "1 row per direct supplier",
    "dim_supplier_tier2_raw":   "1 row per RAW spelling (dirty, pre-resolution)",
    "map_tier1_tier2_declared": "tier1 x metal x declared sub-supplier",
    "dim_contract":             "part x supplier x sourcing role",
    "fact_purchase_orders":     "1 row per PO line   <- lowest grain",
    "fact_supplier_compliance": "1 row per supplier (Tier-1 and Tier-2)",
    "dim_country_risk":         "1 row per country",
    "fact_disruption_events":   "1 row per supply event",
    "_ground_truth_tier2":      "answer key - never read by the matcher",
}

data = {t: load(t) for t in GRAIN}

inventory = pd.DataFrame(
    [{"table": t, "rows": len(df), "cols": df.shape[1], "grain": GRAIN[t]}
     for t, df in data.items()]
)
display(inventory)

# unpack into working variables
dim_metal      = data["dim_metal"]
idx            = data["fact_market_index"]
fx             = data["fact_fx_rate"]
dim_part       = data["dim_part"]
bom            = data["bom_metal_content"]
dim_t1         = data["dim_supplier_tier1"]
raw_t2         = data["dim_supplier_tier2_raw"]
declared       = data["map_tier1_tier2_declared"]
contract       = data["dim_contract"]
po             = data["fact_purchase_orders"]
compliance     = data["fact_supplier_compliance"]
country_risk   = data["dim_country_risk"]
disruptions    = data["fact_disruption_events"]
truth          = data["_ground_truth_tier2"]

po["po_date"] = pd.to_datetime(po["po_date"])
print(f"\nobservation window : {po.po_date.min():%Y-%m-%d} .. {po.po_date.max():%Y-%m-%d}")
print(f"total purchasing spend : EUR {po.net_value_eur.sum():,.0f} across {len(po):,} PO lines")

## 1.2 Referential integrity — prove it, do not assume it

Before a single euro is analysed, every foreign key must resolve. This is the cheapest possible insurance against a dashboard that silently drops 4% of spend.

In [ ]:
checks = [
    ("PO -> part",          set(po.part_id)            <= set(dim_part.part_id)),
    ("PO -> contract",      set(po.contract_id)        <= set(contract.contract_id)),
    ("PO -> tier-1",        set(po.tier1_id)           <= set(dim_t1.tier1_id)),
    ("BOM -> part",         set(bom.part_id)           <= set(dim_part.part_id)),
    ("BOM -> metal",        set(bom.metal_id)          <= set(dim_metal.metal_id)),
    ("contract -> metal",   set(contract.index_metal_id) <= set(dim_metal.metal_id)),
    ("declared -> raw t2",  set(declared.raw_supplier_id) <= set(raw_t2.raw_supplier_id)),
    ("tier-1 -> country",   set(dim_t1.country_code)   <= set(country_risk.country_code)),
    ("tier-2 -> country",   set(raw_t2.country_code)   <= set(country_risk.country_code)),
    ("no ground-truth leak into raw table", "true_entity_id" not in raw_t2.columns),
    ("PO ids unique",       po.po_id.is_unique),
    ("all prices positive", bool((po.unit_price_eur > 0).all())),
]
for label, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
assert all(ok for _, ok in checks), "referential integrity broken"

## 1.3 The hinge of the model: `cost_share`

A purchase order says *what you paid*. It does not say *what metal is inside*. The bill of materials, priced at the index, converts one into the other:

$$\text{metal cost share}_{p,m} = \frac{\text{kg}_{p,m}\times(1+\text{scrap}_{p,m})\times \text{index}_{m,t_0}^{\;\text{EUR/kg}}}{\text{standard cost}_p}$$

Every downstream number — should-cost, cost-at-risk, surcharge justification, Tier-2 exposure — is built on this one column.

In [ ]:
# EUR per kg for every metal and month: index price x FX / 1000
fx_small = fx[["year_month", "currency_code", "eur_per_unit"]]
px = idx.merge(fx_small, left_on=["year_month", "quote_currency"],
               right_on=["year_month", "currency_code"], how="left")
px["eur_per_kg"] = px["settlement_price"] * px["eur_per_unit"] / 1000.0
PK = {(r.metal_id, r.year_month): r.eur_per_kg for r in px.itertuples(index=False)}

BASE_YM = px.year_month.min()
LAST_YM = px.year_month.max()

example = bom[bom.part_id == "P-0001"].merge(
    dim_part[["part_id", "part_description", "standard_cost_eur"]], on="part_id")
display(example[["part_id", "part_description", "metal_id", "kg_per_unit", "scrap_factor",
                 "effective_kg_per_unit", "base_metal_cost_eur", "cost_share",
                 "standard_cost_eur"]])

print(f"\nmetal content of this part = {example.cost_share.sum():.1%} of its standard cost")
print("the remainder is labour, energy, tooling, logistics and the supplier's margin")

---
# Part 2 — The commercial analytics

## 2.1 Scenario S5 — FX turns a copper problem into a bigger copper problem

Procurement teams quote the LME in **USD**. The P&L is reported in **EUR**. When the dollar strengthens during a metal rally, the euro-denominated increase is larger than the headline — and that difference lands entirely in the margin.

In [ ]:
cu = idx[idx.metal_id == "COPPER"].sort_values("month_index").reset_index(drop=True)
usd = fx[fx.currency_code == "USD"].sort_values("month_index").reset_index(drop=True)
cu = cu.merge(usd[["month_index", "eur_per_unit"]], on="month_index")
cu["price_eur_per_t"] = cu.settlement_price * cu.eur_per_unit
cu["idx_usd"] = cu.settlement_price / cu.settlement_price.iloc[0] * 100
cu["idx_eur"] = cu.price_eur_per_t / cu.price_eur_per_t.iloc[0] * 100

fig, ax = plt.subplots()
ax.plot(cu.year_month, cu.idx_usd, label="Copper, USD terms (LME)", linewidth=2)
ax.plot(cu.year_month, cu.idx_eur, label="Copper, EUR terms (what we actually pay)",
        linewidth=2, linestyle="--")
ax.axhline(100, color="grey", linewidth=1)
ax.set_xticks(cu.year_month[::4])
ax.set_title("S5 - the same metal, two currencies (index, month 0 = 100)")
ax.set_ylabel("index")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

a, b = 12, 22   # the ramp window
RAMP_FROM, RAMP_TO = cu.year_month[a], cu.year_month[b]
USD_MOVE = cu.settlement_price[b] / cu.settlement_price[a] - 1
EUR_MOVE = cu.price_eur_per_t[b] / cu.price_eur_per_t[a] - 1
FX_AMPLIFICATION = (1 + EUR_MOVE) / (1 + USD_MOVE) - 1

print(f"copper move {RAMP_FROM} -> {RAMP_TO}")
print(f"   in USD : {USD_MOVE:+.1%}")
print(f"   in EUR : {EUR_MOVE:+.1%}")
print(f"   FX amplification : {FX_AMPLIFICATION:+.1%}")
print("\nA buyer tracking only the LME screen under-states the exposure by this amount.")

## 2.2 Where the money actually sits

Spend by supplier is a purchasing view. **Spend by embedded metal** is a commodity-management view — it is what you hedge, index and negotiate.

In [ ]:
exposure = po.merge(bom[["part_id", "metal_id", "cost_share"]], on="part_id", how="inner")
exposure["metal_spend_eur"] = exposure.net_value_eur * exposure.cost_share

by_metal = (exposure.groupby("metal_id")["metal_spend_eur"].sum()
            .sort_values(ascending=False).reset_index())
by_metal["share_of_metal_spend"] = by_metal.metal_spend_eur / by_metal.metal_spend_eur.sum()
by_metal = by_metal.merge(dim_metal[["metal_id", "criticality", "cbam_in_scope"]], on="metal_id")

total_spend = po.net_value_eur.sum()
total_metal = by_metal.metal_spend_eur.sum()

display(by_metal)
print(f"total purchasing spend .... EUR {total_spend:,.0f}")
print(f"embedded metal value ...... EUR {total_metal:,.0f}  ({total_metal/total_spend:.1%} of spend)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].barh(by_metal.metal_id[::-1], by_metal.metal_spend_eur[::-1] / 1e6)
ax[0].set_title("Embedded metal value (EUR m)")
t1_spend = po.groupby("tier1_id").net_value_eur.sum().sort_values()
ax[1].barh(t1_spend.index, t1_spend.values / 1e6)
ax[1].set_title("Purchasing spend by Tier-1 (EUR m)")
plt.tight_layout()
plt.show()

## 2.3 Scenario S3 — testing a supplier's surcharge claim

One Tier-1 has sent a price letter claiming a **+11.5% increase due to general metal raw material costs**, effective July 2024.

The claim is testable. We know their parts, their bills of material, the index movement and the FX. The should-cost increase is:

$$\Delta_{\text{justified}} = \sum_{m}\ \text{cost share}_{m}\times\left(\frac{\text{index}_{m,\text{after}}}{\text{index}_{m,\text{before}}}-1\right)$$

Anything above that is margin expansion dressed up as a raw-material pass-through.

In [ ]:
T1 = "T1-004"
letter = dim_t1[dim_t1.tier1_id == T1].iloc[0]
claimed = letter.claimed_metal_surcharge_pct / 100
eff_date = pd.Timestamp(letter.surcharge_effective_date)

print(f"supplier   : {letter.tier1_name} ({letter.country_code})")
print(f"claim      : +{claimed:.1%} from {eff_date:%Y-%m-%d}")
print(f"stated as  : {letter.surcharge_justification_text!r}\n")

parts_t1 = po.loc[po.tier1_id == T1, "part_id"].unique()
bom_t1 = bom[bom.part_id.isin(parts_t1)]

before_ym = (eff_date - pd.DateOffset(months=3)).strftime("%Y-%m")
after_ym = (eff_date + pd.DateOffset(months=3)).strftime("%Y-%m")

rows = []
for metal, grp in bom_t1.groupby("metal_id"):
    share = grp.cost_share.mean()
    delta = PK[(metal, after_ym)] / PK[(metal, before_ym)] - 1
    rows.append({"metal_id": metal, "avg_cost_share": share,
                 "index_move": delta, "justified_contribution": share * delta})
just = pd.DataFrame(rows).sort_values("justified_contribution", ascending=False)
display(just)

justified = just.justified_contribution.sum()
gap = claimed - justified

annual_spend_t1 = po[(po.tier1_id == T1) & (po.po_date >= eff_date)].net_value_eur.sum()
months = (po.po_date.max().to_period("M") - eff_date.to_period("M")).n + 1
annualised = annual_spend_t1 / months * 12

print(f"claimed surcharge ................ {claimed:>8.2%}")
print(f"BOM x index justifies ............ {justified:>8.2%}")
print(f"NEGOTIATION GAP .................. {gap:>8.2%}  ({gap*100:.1f} percentage points)")
print(f"\nspend with this supplier since the letter : EUR {annual_spend_t1:,.0f}")
print(f"annualised run-rate ...................... EUR {annualised:,.0f}")
print(f"ANNUAL VALUE OF THE GAP .................. EUR {annualised * gap:,.0f}")
print("\nThat figure is the opening position for the next negotiation round.")

## 2.4 Scenario S1 — the repricing shock that is not in the P&L yet

Copper-dominated parts sitting on **fixed-price** contracts were shielded during the rally. That protection expires. When it does, the supplier reprices to the *current* index in one step.

Cost-at-Risk here is not a forecast — it is arithmetic on contracts we have already signed.

In [ ]:
AS_OF = po.po_date.max()

cu_fixed = contract[(contract.metal_clause_type == "FIXED_PRICE")
                    & (contract.index_metal_id == "COPPER")].copy()
cu_fixed["valid_to"] = pd.to_datetime(cu_fixed.valid_to)
cu_fixed["months_to_expiry"] = (
    (cu_fixed.valid_to.dt.year - AS_OF.year) * 12 + (cu_fixed.valid_to.dt.month - AS_OF.month)
)

last12 = po[po.po_date > AS_OF - pd.DateOffset(months=12)]
spend_at_risk = last12[last12.contract_id.isin(cu_fixed.contract_id)].net_value_eur.sum()

cu_share = (bom[(bom.part_id.isin(cu_fixed.part_id)) & (bom.metal_id == "COPPER")]
            .cost_share.mean())
cu_delta = PK[("COPPER", LAST_YM)] / PK[("COPPER", BASE_YM)] - 1

cost_at_risk = spend_at_risk * cu_share * cu_delta

cu_all = po[po.part_id.isin(contract.loc[contract.index_metal_id == "COPPER", "part_id"])]
print(f"copper-indexed contracts on FIXED_PRICE ....... {len(cu_fixed):>8,}")
print(f"  share of copper spend they cover ............ "
      f"{spend_at_risk / last12[last12.part_id.isin(cu_all.part_id)].net_value_eur.sum():>8.1%}")
print(f"  all expiring within (months) ................ "
      f"{cu_fixed.months_to_expiry.min()} .. {cu_fixed.months_to_expiry.max()}")
print(f"\nannual spend under that protection ............ EUR {spend_at_risk:,.0f}")
print(f"average copper content of those parts ......... {cu_share:>8.1%}")
print(f"copper index move since contract base ......... {cu_delta:>8.1%}")
print(f"\nCOST-AT-RISK on expiry ........................ EUR {cost_at_risk:,.0f} per year")
print(f"as a share of total purchasing spend .......... {cost_at_risk/total_spend:.2%}")
print("\nThis is unbudgeted margin erosion with a known date attached to it.")

display(cu_fixed[["contract_id", "part_id", "tier1_id", "valid_to",
                  "months_to_expiry"]].sort_values("months_to_expiry").head(10))

## 2.5 Scenario S7 — asymmetric clauses leak value when prices fall

A `CAPPED_INDEX` clause caps how much the supplier can pass *up*, but its floor also limits how much we claw *back* when the index falls — and a 3-month reset lag delays even that.

Steel fell over the observation window. So the clause that was negotiated as protection became a cost.

In [ ]:
cap = contract[contract.metal_clause_type == "CAPPED_INDEX"].merge(
    bom[["part_id", "metal_id", "cost_share"]],
    left_on=["part_id", "index_metal_id"], right_on=["part_id", "metal_id"], how="left")
cap["cost_share"] = cap.cost_share.fillna(0)

spend12 = last12.groupby("contract_id").net_value_eur.sum()

rows = []
for c in cap.itertuples(index=False):
    delta = PK[(c.index_metal_id, LAST_YM)] / PK[(c.index_metal_id, BASE_YM)] - 1
    uncapped = c.passthrough_pct * c.cost_share * delta
    actual = float(np.clip(uncapped, c.floor_pct, c.cap_pct))
    s = float(spend12.get(c.contract_id, 0.0))
    rows.append({"contract_id": c.contract_id, "metal": c.index_metal_id,
                 "index_move": delta, "cap_pct": c.cap_pct, "floor_pct": c.floor_pct,
                 "uncapped_adj": uncapped, "clause_adj": actual,
                 "leakage_pct": actual - uncapped, "spend_12m": s,
                 "leakage_eur": (actual - uncapped) * s})
leak = pd.DataFrame(rows)

display(leak[leak.leakage_eur.abs() > 1]
        .sort_values("leakage_eur", ascending=False)
        .head(8)[["contract_id", "metal", "index_move", "floor_pct",
                  "uncapped_adj", "clause_adj", "spend_12m", "leakage_eur"]])

print(f"capped/lagged contracts ................. {len(cap):>8,}")
print(f"of which currently leaking value ........ {int((leak.leakage_eur > 1).sum()):>8,}")
print(f"ANNUAL CLAUSE LEAKAGE ................... EUR {leak.leakage_eur.clip(lower=0).sum():,.0f}")
print("\nActionable recommendation: replace floors with symmetric pass-through")
print("and shorten the reset lag from 3 months to 1 at the next renewal.")

---
# Part 3 — Phase 2: the entity resolution engine

## 3.1 The problem, seen in the raw data

Tier-1 suppliers declare their sub-suppliers through three channels. Nobody maintains a DUNS number. So the same physical smelter arrives three times, spelled three ways.

Look at the raw table below: 48 records, 48 distinct spellings, **zero** usable hard keys.

In [ ]:
print(f"raw Tier-2 records ........ {len(raw_t2)}")
print(f"distinct spellings ........ {raw_t2.raw_supplier_name.nunique()}")
print(f"records with a DUNS ....... {int(raw_t2.duns_number.notna().sum())}   <- no join key exists")
print(f"source systems ............ {sorted(raw_t2.source_system.unique())}\n")

display(raw_t2[raw_t2.country_code == "CN"]
        [["raw_supplier_id", "raw_supplier_name", "supplied_metal_id", "source_system"]]
        .reset_index(drop=True))

## 3.2 Step 1 — Normalisation

The unglamorous stage that does most of the work. Four sub-steps, in a deliberate order:

1. **Remove periods first.** `G.m.b.H.` collapses to `GmbH`, `S.p.A.` to `SpA`. Doing this after general punctuation stripping would shatter them into single letters.
2. **Explicit transliteration, then NFKD.** `ü`→`ue`, `ß`→`ss`, `ı`→`i`, `ş`→`s`, `ł`→`l`. Pure Unicode folding would give `u`, not `ue` — wrong for German.
3. **Strip legal forms.** 60+ of them, from `GmbH` to `Sp. z o.o.`, `d.o.o.`, `A.Ş.`, `Kft`, `UAB`, `FZE`.
4. **Apply domain synonyms.** `aluminum`→`aluminium`, `Int'l`→`international`, `und`→`and`, plus typo repairs seen in supplier surveys.

Two keys are produced: a **token string** and a **compact** whitespace-free string. The second one is what defeats compound-word differences such as `Stahlservice` vs `Stahl Service`.

In [ ]:
from resolve_tier2_entities import (normalise_name, similarity, jaro_winkler,
                                    token_set_ratio, dice, AUTO_MATCH_THRESHOLD,
                                    REVIEW_LOW_THRESHOLD)

demo = raw_t2[raw_t2.raw_supplier_name.str.contains(
    "Yunnan|Donau|Mueller|Müller|MULLER|Anadolu", case=False, regex=True)]
out = demo.copy()
out[["name_norm", "name_compact"]] = pd.DataFrame(
    [normalise_name(n) for n in demo.raw_supplier_name], index=demo.index).values

display(out[["raw_supplier_name", "name_norm", "name_compact"]].reset_index(drop=True))

print("Note how three completely different input strings collapse onto one compact key:")
for n in ["Donau Stahlservice Ges.m.b.H.", "Donau Stahlservice GesmbH", "DONAU STAHL SERVICE GMBH"]:
    print(f"   {n!r:<35} -> {normalise_name(n)[1]!r}")

## 3.3 Step 2 — Blocking

Comparing every record with every other is $O(n^2)$. With 48 records that is 1,128 comparisons; with the ~50,000 sub-suppliers a real n-tier programme at a large manufacturer would hold, it is **1.25 billion**.

So candidates are generated only within `country × metal` blocks. A Swedish copper refinery is never compared with a Chinese aluminium smelter — that comparison is not just wasteful, it is *domain-nonsensical*.

The metric that matters is **blocking recall**: did the blocking key throw away any pair that should have matched?

In [ ]:
from itertools import combinations

recs = raw_t2.copy()
recs[["name_norm", "name_compact"]] = pd.DataFrame(
    [normalise_name(n) for n in recs.raw_supplier_name], index=recs.index).values
records = recs.to_dict("records")

truth_map = dict(zip(truth.raw_supplier_id, truth.true_entity_id))

all_pairs = list(combinations(records, 2))
blocks = {}
for r in records:
    blocks.setdefault((r["country_code"], r["supplied_metal_id"]), []).append(r)
blocked = [p for grp in blocks.values() for p in combinations(grp, 2)]

true_pairs = {frozenset((x, y)) for x, y in combinations(sorted(truth_map), 2)
              if truth_map[x] == truth_map[y]}
kept = {frozenset((r1["raw_supplier_id"], r2["raw_supplier_id"])) for r1, r2 in blocked}

print(f"pairs without blocking ...... {len(all_pairs):>8,}")
print(f"pairs after blocking ........ {len(blocked):>8,}")
print(f"reduction ratio ............. {1 - len(blocked)/len(all_pairs):>8.1%}")
print(f"blocking recall ............. {len(true_pairs & kept)/len(true_pairs):>8.1%}   <- nothing lost")
print(f"\nblocks formed: {len(blocks)}   largest block: {max(len(v) for v in blocks.values())} records")

## 3.4 Step 3 — Two-pass matching

Commercial MDM tools (Informatica, SAP MDG) do not run one algorithm. They run a **deterministic** pass first, then fall back to a **probabilistic** one. This engine does the same:

**Pass 1 — deterministic.** Identical normalised compact key ⇒ certain match, score forced to 1.0.
This was added after the first run produced a *missed merge*: `Donau Stahlservice GesmbH` vs `DONAU STAHL SERVICE GMBH` scored only 0.797, because token-set similarity collapses when a German compound noun is split into two words. The compact keys, however, are byte-identical. **32 of 49 candidate pairs resolve here.**

**Pass 2 — probabilistic ensemble.**

$$\text{score}=0.40\cdot \text{JaroWinkler}(\text{compact}) + 0.35\cdot \text{TokenSet}(\text{norm}) + 0.25\cdot \text{Dice}_{2\text{-gram}}(\text{compact})$$

Each metric covers the others' blind spot:

| Metric | Catches | Blind to |
|---|---|---|
| Jaro-Winkler on compact | transpositions, spacing (`Hong Tai` / `Hongtai`), prefix stability | heavy word reordering |
| Token-set ratio | reordering, extra/missing words | compound splits |
| Bigram Dice | typos (`BAKIRR`), OCR noise | short strings |

In [ ]:
yunnan = [r for r in records if "YUNNAN" in r["raw_supplier_name"].upper()]
other = [r for r in records if "Sunrise" in r["raw_supplier_name"]
         or "SUNRISE" in r["raw_supplier_name"]][:1]

rows = []
for r1, r2 in combinations(yunnan + other, 2):
    s = similarity(r1, r2)
    rows.append({"name_a": r1["raw_supplier_name"][:38],
                 "name_b": r2["raw_supplier_name"][:38],
                 "jaro_winkler": s["jaro_winkler"], "token_set": s["token_set"],
                 "bigram_dice": s["bigram_dice"], "ensemble": s["ensemble_score"],
                 "final": s["match_score"], "rule": s["match_rule"],
                 "truth": "SAME" if truth_map[r1["raw_supplier_id"]] == truth_map[r2["raw_supplier_id"]] else "different"})

demo_scores = pd.DataFrame(rows).sort_values("final", ascending=False)
display(demo_scores)

print(f"auto-merge threshold  : {AUTO_MATCH_THRESHOLD}")
print(f"human-review band     : {REVIEW_LOW_THRESHOLD} .. {AUTO_MATCH_THRESHOLD}")
print("\nThe genuinely different Chinese aluminium firm scores far below the band -")
print("the decision is not a knife-edge call.")

## 3.5 Step 4 — Clustering, survivorship, and the review queue

Edges above the threshold are fed into a **union-find** structure, giving transitive closure: if A matches B and B matches C, all three become one entity even if A and C were never directly compared.

A **golden record** then survives per cluster, by an explicit and auditable rule:

1. properly-cased names beat SHOUTING ERP extracts,
2. the system of record (`ERP_SAP`) beats self-declared portal entries,
3. the longest name wins — it carries the most legal-form detail.

Pairs scoring between 0.78 and 0.90 are **not** merged silently; they go to a review queue for a human buyer. On this dataset the queue is empty, which is itself the finding: no judgement calls were required.

In [ ]:
golden = load("dim_supplier_tier2_golden", PROCESSED)
mapping = load("map_raw_to_golden_tier2", PROCESSED)
pairs_scored = load("er_candidate_pairs", PROCESSED)
sweep = load("er_threshold_sweep", PROCESSED)
review = load("er_review_queue", PROCESSED)

print(f"raw records in ............ {len(mapping)}")
print(f"golden entities out ....... {len(golden)}")
print(f"duplicates eliminated ..... {len(mapping) - len(golden)}")
print(f"pairs needing human review  {len(review)}\n")

display(golden[["t2_entity_id", "golden_name", "country_code", "supplied_metal_id",
                "tier2_role", "n_raw_variants", "source_systems",
                "min_intra_cluster_score"]].head(10))

print("\nexample cluster - every raw spelling that collapsed into one entity:")
big = golden.sort_values("n_raw_variants", ascending=False).iloc[0]
display(mapping[mapping.t2_entity_id == big.t2_entity_id]
        [["raw_supplier_id", "raw_supplier_name", "source_system",
          "name_compact", "survivor_flag"]].reset_index(drop=True))

## 3.6 Step 5 — Evaluation against withheld ground truth

The matcher never reads `_ground_truth_tier2.csv`; only the evaluator does. Three families of metric are reported, because any one alone can mislead:

- **Pairwise precision / recall / F1** — the standard record-linkage measure.
- **Cluster accuracy** — did we reconstruct the *entities*, not just the pairs? A single wrong edge can fuse two correct clusters and still look good pairwise.
- **Separation margin** — the gap between the weakest true match and the strongest false candidate. This is the honest test of whether a threshold is tuned or merely lucky.

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep.threshold, sweep.precision, label="precision", linewidth=2)
ax.plot(sweep.threshold, sweep.recall, label="recall", linewidth=2, linestyle="--")
ax.plot(sweep.threshold, sweep.f1, label="F1", linewidth=2, linestyle=":")
ax.axvspan(sweep.loc[sweep.f1 >= sweep.f1.max() - 1e-9, "threshold"].min(),
           sweep.loc[sweep.f1 >= sweep.f1.max() - 1e-9, "threshold"].max(),
           alpha=0.15, color="green", label="F1 = 1.000 plateau")
ax.axvline(AUTO_MATCH_THRESHOLD, color="black", linewidth=1.2, label="chosen threshold")
ax.set_xlabel("auto-merge threshold")
ax.set_ylim(0, 1.05)
ax.set_title("Threshold sweep - the chosen cut-off sits inside a wide stable plateau")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

pairs_scored["true_same"] = [truth_map[x] == truth_map[y] for x, y in
                             zip(pairs_scored.raw_supplier_id_a, pairs_scored.raw_supplier_id_b)]
weakest_true = pairs_scored.loc[pairs_scored.true_same, "match_score"].min()
strongest_false = pairs_scored.loc[~pairs_scored.true_same, "match_score"].max()

pred = dict(zip(mapping.raw_supplier_id, mapping.t2_entity_id))
tp = fp = fn = 0
for x, y in combinations(sorted(pred), 2):
    ps, ts = pred[x] == pred[y], truth_map[x] == truth_map[y]
    tp += ps and ts
    fp += ps and not ts
    fn += (not ps) and ts
precision = tp / (tp + fp) if tp + fp else 1.0
recall = tp / (tp + fn) if tp + fn else 1.0

print(f"pairwise precision ......... {precision:>8.1%}")
print(f"pairwise recall ............ {recall:>8.1%}")
print(f"pairwise F1 ................ {2*precision*recall/(precision+recall):>8.3f}")
print(f"false merges / missed ...... {fp:>4} / {fn}")
print(f"entities resolved .......... {mapping.t2_entity_id.nunique():>8}  "
      f"(truth: {truth.true_entity_id.nunique()})")
print(f"\nweakest TRUE match ......... {weakest_true:>8.3f}")
print(f"strongest FALSE candidate .. {strongest_false:>8.3f}")
print(f"SEPARATION MARGIN .......... {weakest_true - strongest_false:>8.3f}")
print("\nA margin this wide means the result is not sensitive to threshold choice -")
print("the engine would survive new, unseen supplier names.")

---
# Part 4 — What transparency actually reveals

## 4.1 Scenario S2 — the hidden single source

This is the headline finding. Before resolution the aluminium sub-supplier base looks diversified. After resolution it is one smelter, in one country, behind most of the Tier-1 base.

In [ ]:
decl = declared.merge(mapping[["raw_supplier_id", "t2_entity_id", "golden_name", "country_code"]],
                      on="raw_supplier_id", how="left")

after = (decl.groupby(["t2_entity_id", "golden_name", "country_code"])
         .agg(n_tier1=("tier1_id", "nunique"),
              n_spellings=("declared_supplier_name", "nunique"),
              tier1_list=("tier1_id", lambda s: "|".join(sorted(set(s)))),
              metals=("metal_id", lambda s: "|".join(sorted(set(s)))))
         .reset_index())
after["hidden_by_aliasing"] = (after.n_tier1 > 1) & (after.n_spellings > 1)

hidden = after[after.hidden_by_aliasing]
t1_names = dict(zip(dim_t1.tier1_id, dim_t1.tier1_name))

for h in hidden.itertuples(index=False):
    risk = country_risk[country_risk.country_code == h.country_code].iloc[0]
    print("=" * 78)
    print(f"{h.golden_name}   [{h.country_code} / {h.metals}]")
    print("=" * 78)
    print(f"  declared by {h.n_tier1} of {len(dim_t1)} Tier-1 suppliers, "
          f"spelled {h.n_spellings} different ways\n")
    for t in h.tier1_list.split("|"):
        print(f"     {t}   {t1_names[t]}")
    print("\n  aliases used in the declarations:")
    for nm in sorted(decl.loc[decl.t2_entity_id == h.t2_entity_id,
                              "declared_supplier_name"].unique()):
        print(f"     {nm!r}")
    print(f"\n  country risk profile of {risk.country_name}:")
    print(f"     geopolitical risk index ... {risk.geopolitical_risk_index}")
    print(f"     export control flag ....... {risk.export_control_flag}")
    print(f"     resource nationalism ...... {risk.resource_nationalism_score}")
    ev = disruptions[(disruptions.country_code == h.country_code)
                     & (disruptions.metal_id.isin(h.metals.split("|")))]
    print(f"\n  disruption events already recorded for this country/metal:")
    for e in ev.itertuples(index=False):
        print(f"     {e.start_date}  {e.event_description}  "
              f"(severity {e.severity}, +{e.lead_time_impact_days}d lead time)")
    print(f"\n  BEFORE resolution : {h.n_spellings} apparently independent sub-suppliers")
    print("  AFTER  resolution : 1 entity, 1 country, 1 point of failure")
    print()

### Quantifying it: concentration before vs after resolution

The Herfindahl-Hirschman Index is the standard concentration measure used by competition authorities and by procurement teams for supplier-base risk. Above 2,500 is considered highly concentrated.

Computing it on the **raw declared names** versus on the **resolved entities** shows exactly how much risk the aliasing was hiding.

In [ ]:
t1_metal_spend = (exposure.groupby(["tier1_id", "metal_id"])["metal_spend_eur"]
                  .sum().reset_index())

alloc = (declared.merge(t1_metal_spend, on=["tier1_id", "metal_id"], how="left")
         .merge(mapping[["raw_supplier_id", "t2_entity_id", "golden_name"]],
                on="raw_supplier_id", how="left"))
alloc["allocated_eur"] = alloc.metal_spend_eur * alloc.declared_spend_share


def hhi(series):
    w = series / series.sum()
    return float((w ** 2).sum() * 10_000)


al = alloc[alloc.metal_id == "ALUMINIUM"]
before_h = hhi(al.groupby("declared_supplier_name").allocated_eur.sum())
after_h = hhi(al.groupby("t2_entity_id").allocated_eur.sum())

top = (alloc.groupby(["t2_entity_id", "golden_name"]).allocated_eur.sum()
       .sort_values(ascending=False).reset_index().head(8))
top["share"] = top.allocated_eur / alloc.allocated_eur.sum()
display(top)

print(f"ALUMINIUM sub-supplier concentration (HHI)")
print(f"   as declared, raw names ..... {before_h:>10,.0f}")
print(f"   after entity resolution .... {after_h:>10,.0f}")
print(f"   understatement ............. {after_h - before_h:>10,.0f} points\n")
print(f"single largest Tier-2 exposure : EUR {top.allocated_eur.iloc[0]:,.0f} "
      f"({top.share.iloc[0]:.1%} of all allocated metal value)")
print("\nThe supplier base was never diversified. The data model just could not see it.")

## 4.2 Scenario S4 — cybersecurity exposure is inversely correlated with spend

The job advert names cybersecurity explicitly: *"ensure suppliers comply to security standards, safeguard data exchanges, and reduce risks in digital collaboration."*

Cyber maturity is only meaningful when weighted by exposure. A weak supplier you barely buy from is a nuisance; a weak supplier carrying a fifth of your metal value is a business continuity issue.

In [ ]:
t2_comp = (compliance[compliance.supplier_tier == "Tier-2"]
           .merge(mapping[["raw_supplier_id", "t2_entity_id", "golden_name"]],
                  left_on="supplier_ref_id", right_on="raw_supplier_id", how="left"))
spend_by_entity = alloc.groupby("t2_entity_id").allocated_eur.sum()
t2_comp["allocated_eur"] = t2_comp.t2_entity_id.map(spend_by_entity).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(t2_comp.allocated_eur / 1e6, t2_comp.cyber_maturity_score, s=70)
ax.axhline(50, color="red", linestyle="--", linewidth=1, label="minimum acceptable maturity")
for r in t2_comp.nlargest(3, "allocated_eur").itertuples(index=False):
    ax.annotate(r.golden_name[:28], (r.allocated_eur / 1e6, r.cyber_maturity_score),
                textcoords="offset points", xytext=(8, 4), fontsize=8)
ax.set_xlabel("allocated Tier-2 metal spend (EUR m)")
ax.set_ylabel("cyber maturity score")
ax.set_title("S4 - the biggest sub-suppliers sit below the acceptable line")
ax.legend()
plt.tight_layout()
plt.show()

watch = (t2_comp.nlargest(5, "allocated_eur")
         [["golden_name", "country_code", "allocated_eur", "cyber_maturity_score",
           "iso27001_flag", "cyber_incidents_24m", "audit_overdue_flag", "lksg_risk_class"]])
display(watch)

at_risk = t2_comp.loc[t2_comp.cyber_maturity_score < 50, "allocated_eur"].sum()
print(f"metal value routed through sub-suppliers below the cyber line : "
      f"EUR {at_risk:,.0f}  ({at_risk / alloc.allocated_eur.sum():.1%})")
print(f"suppliers with overdue audits : {int(t2_comp.audit_overdue_flag.sum())} of {len(t2_comp)}")

## 4.3 Scenario S6 — how much of the network can we actually see?

Transparency is a coverage ratio, and it must be weighted by **spend**, not by supplier count. Ten declared small suppliers do not compensate for one undeclared large one.

In [ ]:
covered = declared[["tier1_id", "metal_id"]].drop_duplicates()
cov = t1_metal_spend.merge(covered, on=["tier1_id", "metal_id"], how="left", indicator=True)
cov["is_covered"] = cov._merge == "both"

coverage = cov.loc[cov.is_covered, "metal_spend_eur"].sum() / cov.metal_spend_eur.sum()
blind = cov.loc[~cov.is_covered, "metal_spend_eur"].sum()

by_t1 = (cov.groupby("tier1_id")
         .apply(lambda g: g.loc[g.is_covered, "metal_spend_eur"].sum() / g.metal_spend_eur.sum(),
                include_groups=False)
         .sort_values().rename("coverage").reset_index())
by_t1["tier1_name"] = by_t1.tier1_id.map(t1_names)

fig, ax = plt.subplots()
colors = ["firebrick" if c < 0.6 else "steelblue" for c in by_t1.coverage]
ax.barh(by_t1.tier1_id, by_t1.coverage, color=colors)
ax.axvline(1.0, color="grey", linewidth=1)
ax.set_xlim(0, 1.05)
ax.set_title("S6 - n-tier transparency coverage by Tier-1 (share of metal value declared)")
plt.tight_layout()
plt.show()

display(by_t1[["tier1_id", "tier1_name", "coverage"]])
print(f"overall n-tier transparency coverage ... {coverage:>8.1%}")
print(f"metal value with NO declared Tier-2 .... EUR {blind:,.0f}  ({1-coverage:.1%})")
print("\nThis is the KPI to put on the executive page, with a target and a trend.")

---
# Part 5 — Executive summary

Everything above, reduced to the numbers a commodity-management lead would actually act on.

In [ ]:
bar = "=" * 78
print(bar)
print("GLOBAL 2ND-TIER METALS TRANSPARENCY & COST-AT-RISK ENGINE".center(78))
print(f"as of {AS_OF:%d %B %Y}".center(78))
print(bar)
print(f"{'Purchasing spend analysed':<48} EUR {total_spend:>18,.0f}")
print(f"{'Embedded metal value':<48} EUR {total_metal:>18,.0f}")
print(f"{'PO lines / parts / Tier-1 suppliers':<48} "
      f"{len(po):,} / {len(dim_part)} / {len(dim_t1)}")
print(bar)
print("VALUE IDENTIFIED")
print(f"{'  Unjustified surcharge (S3, annualised)':<48} EUR {annualised*gap:>18,.0f}")
print(f"{'  Cost-at-risk on fixed-price expiry (S1)':<48} EUR {cost_at_risk:>18,.0f}")
print(f"{'  Contract clause leakage (S7, annual)':<48} EUR "
      f"{leak.leakage_eur.clip(lower=0).sum():>18,.0f}")
print(f"{'  TOTAL ADDRESSABLE':<48} EUR "
      f"{annualised*gap + cost_at_risk + leak.leakage_eur.clip(lower=0).sum():>18,.0f}")
print(f"{'  as share of purchasing spend':<48} "
      f"{(annualised*gap + cost_at_risk + leak.leakage_eur.clip(lower=0).sum())/total_spend:>21.2%}")
print(bar)
print("RISK IDENTIFIED")
print(f"{'  n-tier transparency coverage (S6)':<48} {coverage:>21.1%}")
print(f"{'  Hidden single-source entities found (S2)':<48} {len(hidden):>21}")
print(f"{'  Tier-1s depending on that one smelter':<48} "
      f"{int(hidden.n_tier1.max()) if len(hidden) else 0:>21}")
print(f"{'  Metal value behind sub-standard cyber (S4)':<48} EUR {at_risk:>18,.0f}")
print(f"{'  FX amplification of the copper rally (S5)':<48} "
      f"{FX_AMPLIFICATION:>21.1%}")
print(bar)
print("DATA QUALITY")
print(f"{'  Raw Tier-2 records resolved':<48} {len(mapping):>21}")
print(f"{'  Golden entities created':<48} {len(golden):>21}")
print(f"{'  Matching precision / recall':<48} {precision:>10.1%} / {recall:.1%}")
print(f"{'  Pairs requiring human adjudication':<48} {len(review):>21}")
print(bar)

## What each phase proved, and what comes next

| Phase | Status | What it demonstrates to an interviewer |
|---|---|---|
| **1 – Synthetic data generation** | Done | Data *architecture*: star schema, correct grain, derived-not-random prices, self-auditing assertions |
| **2 – Entity resolution** | Done | The "AI bridge": algorithms written from first principles, measured against ground truth, with a human-in-the-loop design |
| **3 – Transformation pipeline** | Next | Gold-layer modelling: `fact_metal_exposure`, should-cost, cost-at-risk and a composite supplier risk score, Pandas locally and PySpark-ready for Fabric |
| **4 – Power BI semantic model + DAX** | Planned | The commercial front end: should-cost variance, unjustified surcharge, n-tier coverage, HHI concentration, clause leakage, FX decomposition |
| **5 – Narrative** | Planned | The five-minute story that turns all of the above into a negotiation mandate |

### The sentence this project exists to earn

> *"Three of your Tier-1 suppliers are not three suppliers. They are one smelter in Yunnan — and the surcharge one of them is charging you is mostly not metal."*